# Sous-clustering des textures MEB

Pour chaque catégorie de texture, sous-clustering K-means sur `block_0` pour révéler des sous-types.  
Chaque sous-type est validé (réel vs artefact d'acquisition) et les patches représentatifs sont affichés.

In [ ]:
import h5py
import numpy as np
import json
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.stats import entropy
from tqdm.auto import tqdm

ROOT     = Path('..').resolve()
DB_PATH  = ROOT / 'data' / 'feature_database' / 'database_meb.h5'
CFG_PATH = ROOT / 'PatchTagger_Output' / 'config' / 'config.json'
IMG_DIR  = ROOT / 'Image_Ouassim'
SEED     = 42
PCA_DIM  = 50
K_RANGE  = [1, 2, 3, 4, 5]
N_SHOW   = 6
CATS_EXCLUDE = [2, 8, 10, 11, 12, 13]

with open(CFG_PATH) as _f:
    _cfg = json.load(_f)
CATEGORIES = {int(k): v['name']
              for k, v in _cfg['available_categories'].items()}

with h5py.File(DB_PATH, 'r') as _h5:
    IMAGE_NAMES  = _h5['metadata/image_names'][:]
    POSITIONS    = _h5['metadata/positions'][:]
    CATEGORY_IDS = _h5['metadata/category_ids'][:]

CATS_VALID = sorted([
    c for c in np.unique(CATEGORY_IDS)
    if c not in CATS_EXCLUDE and (CATEGORY_IDS == c).sum() >= 30
])

KEY = 'block_0'

print(f'ROOT    : {ROOT}')
print(f'DB      : {DB_PATH}')
print(f'Config  : {CFG_PATH}')
print(f'Images  : {IMG_DIR}')
print(f'Catégories valides ({len(CATS_VALID)}) : {CATS_VALID}')
print(f'Patches totaux : {len(CATEGORY_IDS)}')

## Cellule 1 — Trouver le meilleur K par catégorie

In [ ]:
with h5py.File(DB_PATH, 'r') as _h5:
    _X_all = _h5['features'][KEY][:]

# PCA-50d global — fit uniquement sur les patches des catégories valides
_mask_valid = np.isin(CATEGORY_IDS, CATS_VALID)
_pca = PCA(n_components=PCA_DIM, random_state=SEED)
_pca.fit(_X_all[_mask_valid])

subclust_results = {}

print(f'{"Catégorie":<28} │ {"Best K":>6} │ {"Silhouette":>10}')
print('─' * 52)

for _c in tqdm(CATS_VALID, desc='K-means par catégorie'):
    _idx_c = np.where(CATEGORY_IDS == _c)[0]
    _X_c   = _pca.transform(_X_all[_idx_c])           # (N_c, PCA_DIM)

    # L2-normalisation
    _norms = np.linalg.norm(_X_c, axis=1, keepdims=True)
    _X_c   = _X_c / np.where(_norms < 1e-8, 1.0, _norms)

    _sil_scores = {}
    for _k in K_RANGE:
        if _k == 1:
            _sil_scores[_k] = -1          # silhouette indéfinie pour k=1
            continue
        if len(_X_c) < _k:
            _sil_scores[_k] = -1
            continue
        _km     = KMeans(n_clusters=_k, random_state=SEED, n_init=10)
        _labels = _km.fit_predict(_X_c)
        _sil_scores[_k] = silhouette_score(_X_c, _labels)

    _best_k = max(_sil_scores, key=_sil_scores.get)
    # seuil de structure : en dessous → catégorie homogène
    if _sil_scores[_best_k] < 0.15:
        _best_k = 1

    if _best_k > 1:
        _km        = KMeans(n_clusters=_best_k, random_state=SEED, n_init=10)
        _labels    = _km.fit_predict(_X_c)
        _centroids = _km.cluster_centers_
    else:
        _labels    = np.zeros(len(_X_c), dtype=int)
        _centroids = _X_c.mean(axis=0, keepdims=True)

    subclust_results[_c] = {
        'best_k'       : _best_k,
        'silhouettes'  : _sil_scores,
        'labels'       : _labels,
        'centroids'    : _centroids,
        'patch_indices': _idx_c,
        'X_c'          : _X_c,
    }

    _sil_display = _sil_scores[_best_k] if _best_k > 1 else 0.0
    print(f'{CATEGORIES[_c]:<28} │ {_best_k:>6} │ {_sil_display:>10.3f}')

## Cellule 2 — Valider sous-types : réel vs artefact d'acquisition

In [ ]:
print('=== Validation des sous-types ===\n')

for _c in CATS_VALID:
    _res = subclust_results[_c]

    if _res['best_k'] == 1:
        print(f'{CATEGORIES[_c]} : homogène (pas de sous-type)\n')
        continue

    print(f'{CATEGORIES[_c]} → {_res["best_k"]} sous-types :')

    _idx_c  = _res['patch_indices']
    _imgs_c = IMAGE_NAMES[_idx_c]
    _labels = _res['labels']

    _res['verdicts'] = {}

    for _sub in range(_res['best_k']):
        _mask_sub = _labels == _sub
        _n_sub    = int(_mask_sub.sum())
        _imgs_sub = _imgs_c[_mask_sub]

        # Entropie de distribution sur les images sources
        _, _counts = np.unique(_imgs_sub, return_counts=True)
        _probs     = _counts / _counts.sum()
        _H         = entropy(_probs)
        _n_images  = len(_counts)
        _H_max     = np.log(_n_images) if _n_images > 1 else 1.0
        _H_norm    = float(_H / (_H_max + 1e-10))

        if _H_norm > 0.7:
            _verdict = 'réparti → sous-type RÉEL ✅'
        elif _H_norm > 0.4:
            _verdict = 'semi-concentré ⚠️'
        else:
            _verdict = 'concentré → ARTEFACT acquisition ❌'

        _res['verdicts'][_sub] = {
            'n'       : _n_sub,
            'n_images': _n_images,
            'H_norm'  : _H_norm,
            'verdict' : _verdict,
        }

        print(f'  Sous-type {_sub} : {_n_sub} patches · '
              f'{_n_images} images · H={_H_norm:.2f} → {_verdict}')
    print()

## Cellule 3 — Fonctions d'extraction de vignettes

In [ ]:
def get_patch_vignette(_sub_patch_idx):
    """Découpe la région du patch depuis l'image source."""
    _sub_img_name = IMAGE_NAMES[_sub_patch_idx]
    if isinstance(_sub_img_name, (bytes, np.bytes_)):
        _sub_img_name = _sub_img_name.decode()
    _sub_pos = POSITIONS[_sub_patch_idx]   # [x_min, y_min, x_max, y_max]

    _sub_path = IMG_DIR / _sub_img_name
    if not _sub_path.exists():
        return None

    _sub_img = Image.open(_sub_path).convert('L')
    _sub_arr = np.array(_sub_img)

    _sub_x1, _sub_y1, _sub_x2, _sub_y2 = [int(v) for v in _sub_pos]
    return _sub_arr[_sub_y1:_sub_y2, _sub_x1:_sub_x2]


def get_closest_patches(_sub_c, _sub_sub, _sub_n=N_SHOW):
    """N patches globaux les plus proches du sous-centroïde."""
    _sub_res      = subclust_results[_sub_c]
    _sub_idx_c    = _sub_res['patch_indices']
    _sub_X_c      = _sub_res['X_c']
    _sub_labels   = _sub_res['labels']
    _sub_centroid = _sub_res['centroids'][_sub_sub]

    _sub_mask   = _sub_labels == _sub_sub
    _sub_idx    = _sub_idx_c[_sub_mask]
    _sub_X      = _sub_X_c[_sub_mask]

    _sub_dists   = np.linalg.norm(_sub_X - _sub_centroid, axis=1)
    _sub_closest = np.argsort(_sub_dists)[:_sub_n]
    return _sub_idx[_sub_closest]


print('Fonctions get_patch_vignette et get_closest_patches définies.')

## Cellule 4 — Afficher les patches représentatifs par sous-cluster

In [ ]:
for _sub_c in CATS_VALID:
    _sub_res    = subclust_results[_sub_c]
    _sub_best_k = _sub_res['best_k']

    if _sub_best_k == 1:
        continue

    _sub_fig, _sub_axes = plt.subplots(
        _sub_best_k, N_SHOW,
        figsize=(N_SHOW * 1.8, _sub_best_k * 1.9)
    )
    # Garantir 2D même pour best_k == 1
    if _sub_best_k == 1:
        _sub_axes = _sub_axes.reshape(1, -1)
    elif N_SHOW == 1:
        _sub_axes = _sub_axes.reshape(-1, 1)

    for _sub_sub in range(_sub_best_k):
        _sub_closest = get_closest_patches(_sub_c, _sub_sub, N_SHOW)
        _sub_verdict = _sub_res['verdicts'][_sub_sub]

        for _sub_j, _sub_pidx in enumerate(_sub_closest):
            _sub_ax = _sub_axes[_sub_sub, _sub_j]
            try:
                _sub_vign = get_patch_vignette(_sub_pidx)
                if _sub_vign is not None and _sub_vign.size > 0:
                    _sub_ax.imshow(_sub_vign, cmap='gray')
                else:
                    _sub_ax.text(0.5, 0.5, 'N/A', ha='center',
                                 va='center', transform=_sub_ax.transAxes)
            except Exception as _sub_e:
                _sub_ax.text(0.5, 0.5, str(_sub_e)[:20],
                             ha='center', va='center',
                             transform=_sub_ax.transAxes, fontsize=6)
            _sub_ax.axis('off')

            if _sub_j == 0:
                _sub_ax.set_ylabel(
                    f'Sous-type {_sub_sub}\n'
                    f'({_sub_verdict["n"]}p · {_sub_verdict["n_images"]}img)',
                    fontsize=8, rotation=0, ha='right',
                    va='center', labelpad=40
                )

        # Couleur du verdict en annotation latérale
        _sub_color = (
            'green'  if 'RÉEL'     in _sub_verdict['verdict'] else
            'red'    if 'ARTEFACT' in _sub_verdict['verdict'] else
            'orange'
        )
        _sub_axes[_sub_sub, 0].text(
            -0.1, 0.5,
            f'ST{_sub_sub}\nH={_sub_verdict["H_norm"]:.2f}',
            transform=_sub_axes[_sub_sub, 0].transAxes,
            fontsize=8, ha='right', va='center', color=_sub_color
        )

    _sub_cat_name = CATEGORIES[_sub_c]
    _sub_fig.suptitle(
        f'{_sub_cat_name} — {_sub_best_k} sous-types · {KEY}\n'
        'Patches les plus proches de chaque sous-centroïde',
        fontsize=12
    )
    plt.tight_layout()
    _sub_fname = f'subclusters_{_sub_cat_name.replace(" ", "_")}.png'
    plt.savefig(_sub_fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Sauvegardé : {_sub_fname}')

## Cellule 5 — Résumé visuel : silhouette par catégorie

In [ ]:
_sub_fig, _sub_ax = plt.subplots(figsize=(12, 6))

for _sub_c in CATS_VALID:
    _sub_res  = subclust_results[_sub_c]
    _sub_ks   = [k for k in K_RANGE if k > 1]
    _sub_sils = [_sub_res['silhouettes'][k] for k in _sub_ks]
    _sub_color = _cfg['available_categories'][str(_sub_c)].get('color', None)
    _sub_ax.plot(
        _sub_ks, _sub_sils, 'o-', lw=1.8, ms=6,
        color=_sub_color,
        label=f'{CATEGORIES[_sub_c]} (best K={_sub_res["best_k"]})'
    )

_sub_ax.axhline(0.15, color='gray', ls=':', alpha=0.7,
                label='seuil structure (0.15)')
_sub_ax.set_xlabel('Nombre de sous-clusters K', fontsize=11)
_sub_ax.set_ylabel('Silhouette score', fontsize=11)
_sub_ax.set_title(
    'Sous-clustering — Silhouette par catégorie\n'
    f'{KEY} · pic = nombre de sous-types naturels',
    fontsize=12
)
_sub_ax.set_xticks([k for k in K_RANGE if k > 1])
_sub_ax.legend(fontsize=8, loc='best')
_sub_ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('subclustering_silhouettes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé : subclustering_silhouettes.png')